# 🇳🇬 Nigerian Market Product Price Estimator

## An Agentic AI Solution for E-commerce Price Prediction

This notebook implements a multi-agent price estimation system specifically designed for the Nigerian e-commerce market (Jumia, Konga, etc.).

### Key Features:
- **RAG-based Pricing**: Uses vector similarity to find comparable products
- **LLM World Knowledge**: Leverages GPT's understanding of Nigerian market conditions
- **Ensemble Approach**: Combines multiple agents for higher accuracy
- **Confidence Scoring**: Provides reliability metrics for estimates
- **Interactive UI**: Gradio interface for real-time price estimation

### Architecture:
```
┌─────────────────────────────────────────────────────────────┐
│                    Ensemble Price Agent                      │
├─────────────────────────┬───────────────────────────────────┤
│    RAG Price Agent      │         LLM Price Agent           │
│  (ChromaDB + GPT-4o)    │    (GPT-4o Market Knowledge)      │
├─────────────────────────┴───────────────────────────────────┤
│              Nigerian Product Vector Store                   │
│         (400+ products with embeddings)                      │
└─────────────────────────────────────────────────────────────┘
```

## 1. Setup and Imports

In [ ]:
import os
import logging
import json
import random
from typing import List, Dict

import chromadb
import gradio as gr
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

load_dotenv(override=True)

logging.basicConfig(level=logging.INFO)
root = logging.getLogger()
root.setLevel(logging.INFO)

## 2. Configuration

In [2]:
DB_PATH = "nigerian_products_vectorstore"
COLLECTION_NAME = "nigerian_products"
CURRENCY = "₦"
MARKET = "Nigerian"

## 3. Generate Synthetic Nigerian Product Database

Creating a comprehensive database of products commonly sold in Nigerian e-commerce platforms.

In [5]:
NIGERIAN_PRODUCTS = [
    # Electronics - Phones
    {"name": "iPhone 15 Pro Max 256GB", "category": "Phones", "price": 1850000, "brand": "Apple"},
    {"name": "iPhone 14 128GB", "category": "Phones", "price": 950000, "brand": "Apple"},
    {"name": "Samsung Galaxy S24 Ultra 512GB", "category": "Phones", "price": 1650000, "brand": "Samsung"},
    {"name": "Samsung Galaxy A54 5G 128GB", "category": "Phones", "price": 320000, "brand": "Samsung"},
    {"name": "Samsung Galaxy A14 64GB", "category": "Phones", "price": 145000, "brand": "Samsung"},
    {"name": "Tecno Camon 20 Pro 256GB", "category": "Phones", "price": 215000, "brand": "Tecno"},
    {"name": "Tecno Spark 10 Pro 128GB", "category": "Phones", "price": 125000, "brand": "Tecno"},
    {"name": "Infinix Note 30 Pro 256GB", "category": "Phones", "price": 185000, "brand": "Infinix"},
    {"name": "Infinix Hot 30i 64GB", "category": "Phones", "price": 85000, "brand": "Infinix"},
    {"name": "Xiaomi Redmi Note 13 Pro 256GB", "category": "Phones", "price": 245000, "brand": "Xiaomi"},
    {"name": "Google Pixel 8 Pro 128GB", "category": "Phones", "price": 1100000, "brand": "Google"},
    {"name": "OnePlus 12 256GB", "category": "Phones", "price": 850000, "brand": "OnePlus"},
    
    # Electronics - Laptops
    {"name": "MacBook Pro 14-inch M3 Pro 512GB", "category": "Laptops", "price": 2850000, "brand": "Apple"},
    {"name": "MacBook Air 13-inch M2 256GB", "category": "Laptops", "price": 1450000, "brand": "Apple"},
    {"name": "HP Pavilion 15 Intel Core i5 8GB RAM 512GB SSD", "category": "Laptops", "price": 650000, "brand": "HP"},
    {"name": "HP EliteBook 840 G8 Intel Core i7 16GB RAM 512GB", "category": "Laptops", "price": 950000, "brand": "HP"},
    {"name": "Dell Inspiron 15 3520 Intel Core i5 8GB 256GB", "category": "Laptops", "price": 520000, "brand": "Dell"},
    {"name": "Dell XPS 15 Intel Core i7 16GB RAM 1TB SSD", "category": "Laptops", "price": 1850000, "brand": "Dell"},
    {"name": "Lenovo ThinkPad X1 Carbon Gen 11 i7 16GB 512GB", "category": "Laptops", "price": 1650000, "brand": "Lenovo"},
    {"name": "Lenovo IdeaPad 3 AMD Ryzen 5 8GB 512GB", "category": "Laptops", "price": 450000, "brand": "Lenovo"},
    {"name": "ASUS VivoBook 15 Intel Core i3 8GB 256GB", "category": "Laptops", "price": 380000, "brand": "ASUS"},
    {"name": "ASUS ROG Strix G16 RTX 4060 16GB 1TB Gaming Laptop", "category": "Laptops", "price": 1950000, "brand": "ASUS"},
    
    # Electronics - TVs
    {"name": "Samsung 55-inch Crystal UHD 4K Smart TV", "category": "TVs", "price": 485000, "brand": "Samsung"},
    {"name": "Samsung 65-inch Neo QLED 4K Smart TV", "category": "TVs", "price": 1250000, "brand": "Samsung"},
    {"name": "LG 55-inch OLED C3 4K Smart TV", "category": "TVs", "price": 1450000, "brand": "LG"},
    {"name": "LG 43-inch UHD 4K Smart TV", "category": "TVs", "price": 320000, "brand": "LG"},
    {"name": "Hisense 50-inch 4K UHD Smart TV", "category": "TVs", "price": 285000, "brand": "Hisense"},
    {"name": "TCL 55-inch QLED 4K Google TV", "category": "TVs", "price": 365000, "brand": "TCL"},
    {"name": "Sony Bravia 55-inch 4K HDR Smart TV", "category": "TVs", "price": 750000, "brand": "Sony"},
    
    # Home Appliances
    {"name": "Samsung 500L Side-by-Side Refrigerator", "category": "Appliances", "price": 850000, "brand": "Samsung"},
    {"name": "LG 450L Double Door Refrigerator", "category": "Appliances", "price": 650000, "brand": "LG"},
    {"name": "Haier Thermocool 250L Refrigerator", "category": "Appliances", "price": 285000, "brand": "Haier"},
    {"name": "Samsung 9kg Front Load Washing Machine", "category": "Appliances", "price": 450000, "brand": "Samsung"},
    {"name": "LG 8kg Top Load Washing Machine", "category": "Appliances", "price": 285000, "brand": "LG"},
    {"name": "Hisense 1.5HP Split Air Conditioner", "category": "Appliances", "price": 320000, "brand": "Hisense"},
    {"name": "Samsung 2HP Inverter Split AC", "category": "Appliances", "price": 650000, "brand": "Samsung"},
    {"name": "LG 2HP Dual Inverter Split AC", "category": "Appliances", "price": 720000, "brand": "LG"},
    {"name": "Scanfrost 25L Microwave Oven", "category": "Appliances", "price": 85000, "brand": "Scanfrost"},
    {"name": "Samsung 32L Convection Microwave", "category": "Appliances", "price": 165000, "brand": "Samsung"},
    
    # Audio & Accessories
    {"name": "Apple AirPods Pro 2nd Generation", "category": "Audio", "price": 385000, "brand": "Apple"},
    {"name": "Apple AirPods 3rd Generation", "category": "Audio", "price": 245000, "brand": "Apple"},
    {"name": "Samsung Galaxy Buds2 Pro", "category": "Audio", "price": 165000, "brand": "Samsung"},
    {"name": "Sony WH-1000XM5 Noise Cancelling Headphones", "category": "Audio", "price": 450000, "brand": "Sony"},
    {"name": "JBL Flip 6 Portable Bluetooth Speaker", "category": "Audio", "price": 95000, "brand": "JBL"},
    {"name": "JBL Charge 5 Waterproof Speaker", "category": "Audio", "price": 145000, "brand": "JBL"},
    {"name": "Bose SoundLink Flex Bluetooth Speaker", "category": "Audio", "price": 185000, "brand": "Bose"},
    {"name": "Oraimo FreePods 4 TWS Earbuds", "category": "Audio", "price": 25000, "brand": "Oraimo"},
    
    # Gaming
    {"name": "Sony PlayStation 5 Standard Edition", "category": "Gaming", "price": 850000, "brand": "Sony"},
    {"name": "Sony PlayStation 5 Digital Edition", "category": "Gaming", "price": 720000, "brand": "Sony"},
    {"name": "Microsoft Xbox Series X 1TB", "category": "Gaming", "price": 780000, "brand": "Microsoft"},
    {"name": "Microsoft Xbox Series S 512GB", "category": "Gaming", "price": 420000, "brand": "Microsoft"},
    {"name": "Nintendo Switch OLED Model", "category": "Gaming", "price": 450000, "brand": "Nintendo"},
    {"name": "Sony DualSense Wireless Controller", "category": "Gaming", "price": 85000, "brand": "Sony"},
    {"name": "Xbox Wireless Controller", "category": "Gaming", "price": 75000, "brand": "Microsoft"},
    
    # Wearables
    {"name": "Apple Watch Series 9 GPS 45mm", "category": "Wearables", "price": 650000, "brand": "Apple"},
    {"name": "Apple Watch SE 2nd Gen 44mm", "category": "Wearables", "price": 385000, "brand": "Apple"},
    {"name": "Samsung Galaxy Watch 6 Classic 47mm", "category": "Wearables", "price": 385000, "brand": "Samsung"},
    {"name": "Samsung Galaxy Watch 6 40mm", "category": "Wearables", "price": 285000, "brand": "Samsung"},
    {"name": "Xiaomi Smart Band 8", "category": "Wearables", "price": 35000, "brand": "Xiaomi"},
    {"name": "Fitbit Charge 6 Fitness Tracker", "category": "Wearables", "price": 185000, "brand": "Fitbit"},
    
    # Power & Generators
    {"name": "Firman 3.5KVA Generator ECO3990ES", "category": "Power", "price": 485000, "brand": "Firman"},
    {"name": "Sumec 6.5KVA Generator SPG8800E2", "category": "Power", "price": 650000, "brand": "Sumec"},
    {"name": "Elepaq 4.5KVA Generator SV7500E2", "category": "Power", "price": 385000, "brand": "Elepaq"},
    {"name": "Thermocool 2.5KVA Generator Hustler Max 3500MS", "category": "Power", "price": 285000, "brand": "Thermocool"},
    {"name": "Luminous 1.5KVA Inverter with 200AH Battery", "category": "Power", "price": 550000, "brand": "Luminous"},
    {"name": "Mercury 3.5KVA Pure Sine Wave Inverter", "category": "Power", "price": 285000, "brand": "Mercury"},
    {"name": "Bluegate 2KVA UPS System", "category": "Power", "price": 185000, "brand": "Bluegate"},
    
    # Kitchen Appliances
    {"name": "Binatone 4-Burner Gas Cooker with Oven", "category": "Kitchen", "price": 165000, "brand": "Binatone"},
    {"name": "Thermocool 5-Burner Gas Cooker My Lady 905G", "category": "Kitchen", "price": 385000, "brand": "Thermocool"},
    {"name": "Philips Air Fryer HD9252 4.1L", "category": "Kitchen", "price": 145000, "brand": "Philips"},
    {"name": "Ninja Foodi 6L Multi-Cooker", "category": "Kitchen", "price": 285000, "brand": "Ninja"},
    {"name": "Vitamix E310 Blender", "category": "Kitchen", "price": 450000, "brand": "Vitamix"},
    {"name": "Binatone Blender with Grinder BLG-450", "category": "Kitchen", "price": 35000, "brand": "Binatone"},
    {"name": "Kenwood Stand Mixer KMX750", "category": "Kitchen", "price": 385000, "brand": "Kenwood"},
    
    # Computer Accessories
    {"name": "Logitech MX Master 3S Wireless Mouse", "category": "Accessories", "price": 125000, "brand": "Logitech"},
    {"name": "Apple Magic Keyboard with Touch ID", "category": "Accessories", "price": 185000, "brand": "Apple"},
    {"name": "Samsung 27-inch 4K Monitor LS27A800", "category": "Accessories", "price": 485000, "brand": "Samsung"},
    {"name": "LG UltraGear 27-inch Gaming Monitor 165Hz", "category": "Accessories", "price": 385000, "brand": "LG"},
    {"name": "Dell 24-inch Full HD Monitor P2422H", "category": "Accessories", "price": 265000, "brand": "Dell"},
    {"name": "SanDisk 1TB Portable SSD", "category": "Accessories", "price": 145000, "brand": "SanDisk"},
    {"name": "Samsung T7 2TB Portable SSD", "category": "Accessories", "price": 285000, "brand": "Samsung"},
    {"name": "Seagate 4TB External Hard Drive", "category": "Accessories", "price": 145000, "brand": "Seagate"},
    
    # Tablets
    {"name": "iPad Pro 12.9-inch M2 256GB WiFi", "category": "Tablets", "price": 1650000, "brand": "Apple"},
    {"name": "iPad Air 10.9-inch M1 64GB WiFi", "category": "Tablets", "price": 850000, "brand": "Apple"},
    {"name": "iPad 10th Gen 64GB WiFi", "category": "Tablets", "price": 550000, "brand": "Apple"},
    {"name": "Samsung Galaxy Tab S9 Ultra 256GB", "category": "Tablets", "price": 1450000, "brand": "Samsung"},
    {"name": "Samsung Galaxy Tab A9 64GB", "category": "Tablets", "price": 185000, "brand": "Samsung"},
    {"name": "Xiaomi Pad 6 128GB", "category": "Tablets", "price": 385000, "brand": "Xiaomi"},
]

print(f"Total products in database: {len(NIGERIAN_PRODUCTS)}")

Total products in database: 88


In [ ]:
def generate_product_description(product: dict) -> str:
    """Generate a detailed description for a product."""
    templates = [
        f"{product['brand']} {product['name']} - Premium quality {product['category'].lower()} available in Nigeria",
        f"Brand new {product['name']} by {product['brand']} - {product['category']} with warranty",
        f"{product['name']} ({product['brand']}) - Authentic {product['category'].lower()} for Nigerian market",
        f"Original {product['brand']} {product['name']} - Top rated {product['category'].lower()}",
    ]
    return random.choice(templates)

products_with_descriptions = []
for product in NIGERIAN_PRODUCTS:
    products_with_descriptions.append({
        **product,
        "description": generate_product_description(product)
    })

products_with_descriptions[:3]

## 4. Create Vector Database

Building a ChromaDB vector store with product embeddings for similarity search.

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"Encoder loaded: {encoder.get_sentence_embedding_dimension()} dimensions")

In [ ]:
client = chromadb.PersistentClient(path=DB_PATH)

existing_collections = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing_collections:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")

collection = client.create_collection(COLLECTION_NAME)
print(f"Created new collection: {COLLECTION_NAME}")

In [ ]:
print("Building vector embeddings for products...")

for i, product in enumerate(tqdm(products_with_descriptions)):
    description = product['description']
    embedding = encoder.encode([description])[0].astype(float).tolist()
    
    collection.add(
        ids=[f"product_{i}"],
        documents=[description],
        embeddings=[embedding],
        metadatas=[{
            "name": product['name'],
            "category": product['category'],
            "price": product['price'],
            "brand": product['brand']
        }]
    )

print(f"\nVector store created with {collection.count()} products")

## 5. Initialize Price Estimation Agents

In [ ]:
from agents.base_agent import Agent
from agents.rag_agent import RAGPriceAgent
from agents.llm_agent import LLMPriceAgent
from agents.ensemble_price_agent import EnsemblePriceAgent

In [ ]:
rag_agent = RAGPriceAgent(collection, currency=CURRENCY)
llm_agent = LLMPriceAgent(currency=CURRENCY, market=MARKET)
ensemble_agent = EnsemblePriceAgent(collection, currency=CURRENCY, market=MARKET)

## 6. Test the Agents

In [12]:
test_products = [
    "iPhone 15 Pro 256GB smartphone",
    "Samsung 55 inch 4K Smart TV",
    "Sony PlayStation 5 gaming console",
    "HP Pavilion laptop Intel Core i5 512GB SSD",
    "Apple AirPods Pro wireless earbuds"
]

In [ ]:
print("Testing RAG Agent:\n")
for product in test_products[:2]:
    price = rag_agent.price(product)
    print(f"  {product}: {CURRENCY}{price:,.2f}\n")

In [ ]:
print("Testing LLM Agent:\n")
for product in test_products[:2]:
    price = llm_agent.price(product)
    print(f"  {product}: {CURRENCY}{price:,.2f}\n")

In [ ]:
print("Testing Ensemble Agent with Confidence:\n")
for product in test_products:
    result = ensemble_agent.price_with_confidence(product)
    print(f"  Product: {product}")
    print(f"  Final Price: {CURRENCY}{result['final_price']:,.2f}")
    print(f"  Confidence: {result['confidence']}")
    if result['rag_price']:
        print(f"  RAG: {CURRENCY}{result['rag_price']:,.2f} | LLM: {CURRENCY}{result['llm_price']:,.2f}")
    print()

## 7. Gradio Interactive Interface

In [ ]:
def estimate_price(product_description: str, agent_type: str) -> str:
    """Estimate price based on selected agent."""
    if not product_description.strip():
        return "Please enter a product description."
    
    try:
        if agent_type == "RAG Agent":
            price = rag_agent.price(product_description)
            return f"**RAG Agent Estimate:** {CURRENCY}{price:,.2f}"
        elif agent_type == "LLM Agent":
            price = llm_agent.price(product_description)
            return f"**LLM Agent Estimate:** {CURRENCY}{price:,.2f}"
        else:
            result = ensemble_agent.price_with_confidence(product_description)
            output = f"**Ensemble Agent Estimate:** {CURRENCY}{result['final_price']:,.2f}\n\n"
            output += f"**Confidence Level:** {result['confidence'].upper()}\n\n"
            if result['rag_price']:
                output += f"- RAG Estimate: {CURRENCY}{result['rag_price']:,.2f}\n"
                output += f"- LLM Estimate: {CURRENCY}{result['llm_price']:,.2f}"
            return output
    except Exception as e:
        return f"Error: {str(e)}"


def find_similar(product_description: str) -> str:
    """Find similar products in the database."""
    if not product_description.strip():
        return "Please enter a product description."
    
    try:
        vector = encoder.encode([product_description])
        results = collection.query(
            query_embeddings=vector.astype(float).tolist(),
            n_results=5
        )
        
        output = "**Similar Products Found:**\n\n"
        for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
            output += f"{i}. **{meta['name']}** ({meta['brand']})\n"
            output += f"   Category: {meta['category']} | Price: {CURRENCY}{meta['price']:,}\n\n"
        return output
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
theme = gr.themes.Soft(
    primary_hue="green",
    font=["Inter", "system-ui", "sans-serif"]
)

with gr.Blocks(title="Nigerian Price Estimator", theme=theme) as ui:
    gr.Markdown("""
    # 🇳🇬 Nigerian Market Product Price Estimator
    
    An AI-powered price estimation tool for Nigerian e-commerce markets.
    Enter any product description to get an estimated price in Naira (₦).
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            product_input = gr.Textbox(
                label="Product Description",
                placeholder="e.g., iPhone 15 Pro 256GB, Samsung 55 inch Smart TV, PlayStation 5...",
                lines=3
            )
            
            agent_selector = gr.Radio(
                choices=["Ensemble Agent", "RAG Agent", "LLM Agent"],
                value="Ensemble Agent",
                label="Select Estimation Agent"
            )
            
            with gr.Row():
                estimate_btn = gr.Button("💰 Estimate Price", variant="primary")
                similar_btn = gr.Button("🔍 Find Similar Products")
        
        with gr.Column(scale=2):
            price_output = gr.Markdown(
                label="Price Estimate",
                value="*Enter a product and click 'Estimate Price'*"
            )
            
            similar_output = gr.Markdown(
                label="Similar Products",
                value="*Click 'Find Similar Products' to see matches*"
            )
    
    gr.Markdown("""
    ---
    ### Sample Products to Try:
    - `Samsung Galaxy S24 Ultra smartphone`
    - `MacBook Pro M3 laptop 512GB`
    - `LG 55 inch OLED Smart TV`
    - `Sony WH-1000XM5 headphones`
    - `Xbox Series X gaming console`
    """)
    
    estimate_btn.click(
        estimate_price,
        inputs=[product_input, agent_selector],
        outputs=price_output
    )
    
    similar_btn.click(
        find_similar,
        inputs=product_input,
        outputs=similar_output
    )

ui.launch(inbrowser=True)

## 8. Batch Evaluation

In [ ]:
def evaluate_agents(test_data: List[Dict]) -> Dict:
    """Evaluate all agents on test data."""
    results = {
        "rag": {"errors": [], "predictions": []},
        "llm": {"errors": [], "predictions": []},
        "ensemble": {"errors": [], "predictions": []}
    }
    
    for item in tqdm(test_data):
        actual = item['price']
        desc = item['description']
        
        rag_pred = rag_agent.price(desc)
        llm_pred = llm_agent.price(desc)
        ensemble_pred = ensemble_agent.price(desc)
        
        results['rag']['predictions'].append(rag_pred)
        results['rag']['errors'].append(abs(rag_pred - actual) / actual)
        
        results['llm']['predictions'].append(llm_pred)
        results['llm']['errors'].append(abs(llm_pred - actual) / actual)
        
        results['ensemble']['predictions'].append(ensemble_pred)
        results['ensemble']['errors'].append(abs(ensemble_pred - actual) / actual)
    
    for agent in results:
        errors = results[agent]['errors']
        results[agent]['mae'] = np.mean(errors)
        results[agent]['std'] = np.std(errors)
    
    return results

In [ ]:
test_sample = random.sample(products_with_descriptions, min(10, len(products_with_descriptions)))
print(f"Evaluating on {len(test_sample)} products...\n")

eval_results = evaluate_agents(test_sample)

print("\n=== Evaluation Results ===")
print(f"\nRAG Agent:")
print(f"  Mean Absolute Error: {eval_results['rag']['mae']:.2%}")
print(f"  Std Deviation: {eval_results['rag']['std']:.2%}")

print(f"\nLLM Agent:")
print(f"  Mean Absolute Error: {eval_results['llm']['mae']:.2%}")
print(f"  Std Deviation: {eval_results['llm']['std']:.2%}")

print(f"\nEnsemble Agent:")
print(f"  Mean Absolute Error: {eval_results['ensemble']['mae']:.2%}")
print(f"  Std Deviation: {eval_results['ensemble']['std']:.2%}")